In [49]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [50]:
PDF_PATH = "example.pdf"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "gemma4:31b-cloud"
CHROMA_DIR = "./chroma_db"      
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4

In [51]:
docs = PyPDFLoader("example.pdf").load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)
print(f"Loaded {len(docs)} pages, split into {len(chunks)} chunks.")


Loaded 26 pages, split into 74 chunks.


In [52]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [53]:
llm = ChatOllama(model=LLM_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question using only the context below. "
    "If the context does not contain the answer, say you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [56]:

while True:
    question = input("\nAsk a question (or 'quit'): ").strip()
    if question.lower() in {"quit", "exit", "q"}:
        break
    print("\nMe :", question)
    print("\nGemma:", chain.invoke(question))



Me : hi

Gemma: Hello! How can I help you?

Me : filter links from pdf

Gemma: I don't know.

Me : give all links from pdf

Gemma: I don't know.

Me : what is scope

Gemma: The provided context does not define what scope is; it only lists "Determining the scope of the information security management system" as section 4.3 on page 2.

Me : reviewing the nonconformity?

Gemma: Reviewing the nonconformity is one of the steps an organization shall take to evaluate the need for action to eliminate the causes of nonconformity, in order that it does not recur or occur elsewhere.

Me : 

Gemma: You didn't provide a question, and the context does not contain any information. I don't know.
